# Tracepad guided demo

Run the saved analysis turns from top to bottom. Each result becomes a named, inspectable object that the next turn can reference.

In [ ]:
from pathlib import Path
import pandas as pd

orders = pd.read_csv(Path("demo/data/retail_orders.csv"), parse_dates=["order_date"])
print(f"{len(orders):,} rows x {len(orders.columns)} columns")
print(orders.isna().sum().loc[lambda values: values > 0])
orders

In [ ]:
monthly_revenue = (
    orders.assign(month=orders["order_date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "channel"], as_index=False)
    .agg(revenue=("revenue", "sum"), order_count=("order_id", "count"))
)
monthly_revenue["average_order_value"] = monthly_revenue["revenue"] / monthly_revenue["order_count"]
monthly_revenue

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.8))
for channel, values in monthly_revenue.groupby("channel"):
    ax.plot(values["month"], values["revenue"], marker="o", linewidth=2, label=channel)
ax.set(title="Monthly retail revenue by channel", xlabel="Month", ylabel="Revenue")
ax.legend(title="Channel", frameon=False)
ax.grid(axis="y", alpha=0.2)
fig.autofmt_xdate()
fig.tight_layout()
fig

In [ ]:
import statsmodels.formula.api as smf

return_model = smf.logit(
    "returned ~ discount + unit_price + delivery_days + C(channel) + C(category)",
    data=orders,
).fit(disp=False)
return_model

In [ ]:
return_predictions = orders.sample(20, random_state=12).copy()
return_predictions["predicted_return_probability"] = return_model.predict(return_predictions)
return_predictions = return_predictions[
    ["order_id", "channel", "category", "discount", "delivery_days", "returned", "predicted_return_probability"]
].sort_values("predicted_return_probability", ascending=False)
return_predictions